<a href="https://colab.research.google.com/github/rfcastrovera/BIGDATA/blob/main/Lab2_Benchmark_Spark_DuckDB_Polars.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 2 — Benchmark: Spark vs DuckDB vs Polars
**Análisis de Big Data · Magíster en Data Science UDD · Sesión 2 (vie 14 ago)**

**Alumno Ricardo Castro Vera**

**Objetivos:** ejecutar la **misma consulta con funciones de ventana** en tres motores (Spark, DuckDB, Polars), **leer e interpretar el plan de ejecución** (las 4 señales de la clase) y producir una **recomendación fundamentada en números** — el mismo ejercicio que harán en la Fase 1 con su propio dataset.

**Entorno:** Google Colab (ejecuta el setup) o Databricks CE (salta el `pip install` de pyspark).

**Dataset:** NYC Yellow Taxi — **año completo 2023** (~38M viajes, 12 archivos Parquet, ~700 MB).

**Entrega:** al final del bloque, notebook ejecutado de arriba a abajo. **Cuenta para el 25% de laboratorios.**

**Rúbrica (la de la lámina):** corre de arriba a abajo ✓ · plan interpretado correctamente ✓ · recomendación con números ✓ · uso de IA declarado ✓

## 0. Setup

In [1]:
# Colab / Jupyter local (en Databricks CE, instalar solo duckdb y polars)
%pip install -q pyspark==3.5.4 duckdb polars


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.3/317.3 MB 1.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 13.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.4 which is incompatible.


In [2]:
# Descarga de los 12 meses de 2023 (~700 MB en total; toma unos minutos).
# Si el tiempo o la RAM aprietan, baja MESES a 6 — pero anótalo en tu recomendación:
# el tamaño de los datos ES parte de la conclusión.
import urllib.request, os, glob
MESES = 12
for m in range(1, MESES + 1):
    f = f"yellow_tripdata_2023-{m:02d}.parquet"
    if not os.path.exists(f):
        urllib.request.urlretrieve(f"https://d37ci6vzurychx.cloudfront.net/trip-data/{f}", f)
        print("ok", f)
ARCHIVOS = sorted(glob.glob("yellow_tripdata_2023-*.parquet"))
print(len(ARCHIVOS), "archivos ·", round(sum(os.path.getsize(a) for a in ARCHIVOS)/1e6), "MB")


ok yellow_tripdata_2023-01.parquet
ok yellow_tripdata_2023-02.parquet
ok yellow_tripdata_2023-03.parquet
ok yellow_tripdata_2023-04.parquet
ok yellow_tripdata_2023-05.parquet
ok yellow_tripdata_2023-06.parquet
ok yellow_tripdata_2023-07.parquet
ok yellow_tripdata_2023-08.parquet
ok yellow_tripdata_2023-09.parquet
ok yellow_tripdata_2023-10.parquet
ok yellow_tripdata_2023-11.parquet
ok yellow_tripdata_2023-12.parquet
12 archivos · 636 MB


In [3]:
# Cronómetro del benchmark: 3 corridas, nos quedamos con la mediana
# (la 1a corrida paga cachés y arranque — también es un dato, guárdala aparte).
import time, statistics
resultados = {}
def cronometrar(motor, fn, corridas=3):
    tiempos = []
    for _ in range(corridas):
        t0 = time.perf_counter(); fn(); tiempos.append(time.perf_counter() - t0)
    resultados[motor] = {"fria": round(tiempos[0], 2), "mediana": round(statistics.median(tiempos), 2)}
    print(motor, resultados[motor])


## 1. La consulta de referencia

Una sola pregunta de negocio, idéntica en los tres motores:

> **Por zona de origen: los 3 días de mayor demanda del año y la media móvil de 7 días de viajes.**

Usa lo visto en el Bloque 1: `row_number`/`rank` (ranking), `avg over rows between` (móvil). Regla del benchmark: **misma lógica, mismos filtros, mismos datos** — si un motor hace menos trabajo, la comparación no vale.

In [4]:
# SQL de referencia (Spark y DuckDB lo ejecutan tal cual)
SQL = """
WITH viajes_diarios AS (
  SELECT PULocationID AS zona,
         CAST(tpep_pickup_datetime AS DATE) AS fecha,
         COUNT(*) AS viajes
  FROM viajes
  WHERE total_amount > 0 AND trip_distance > 0
    AND tpep_pickup_datetime >= '2023-01-01' AND tpep_pickup_datetime < '2024-01-01'
  GROUP BY 1, 2
)
SELECT * FROM (
  SELECT zona, fecha, viajes,
         ROW_NUMBER() OVER (PARTITION BY zona ORDER BY viajes DESC) AS pos,
         ROUND(AVG(viajes) OVER (PARTITION BY zona ORDER BY fecha
               ROWS BETWEEN 6 PRECEDING AND CURRENT ROW), 1) AS media_movil_7d
  FROM viajes_diarios)
WHERE pos <= 3
ORDER BY zona, pos
"""


## 2. Motor 1: Spark

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from functools import reduce

spark = (SparkSession.builder.appName("ADBD-Lab2")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())

# Ojo: los Parquet de TLC cambian tipos y nombres de columna entre meses
# (p. ej. airport_fee / Airport_fee). Leer los 12 archivos juntos revienta en ejecución.
# Arreglo: leer cada archivo con SU esquema, quedarnos solo con las 4 columnas del query
# y unificar tipos. (Primer hallazgo del lab: la calidad del dato también se benchmarkea.)
COLS = {"PULocationID": "bigint", "tpep_pickup_datetime": "timestamp",
        "total_amount": "double", "trip_distance": "double"}
partes = [spark.read.parquet(a).select([F.col(c).cast(t).alias(c) for c, t in COLS.items()])
          for a in ARCHIVOS]
reduce(lambda a, b: a.unionByName(b), partes).createOrReplaceTempView("viajes")
spark.sql("SELECT COUNT(*) AS filas FROM viajes").show()


+--------+
|   filas|
+--------+
|38310226|
+--------+



In [6]:
cronometrar("spark", lambda: spark.sql(SQL).collect())
spark.sql(SQL).show(6)


spark {'fria': 37.68, 'mediana': 27.46}
+----+----------+------+---+--------------+
|zona|     fecha|viajes|pos|media_movil_7d|
+----+----------+------+---+--------------+
|   1|2023-07-09|    11|  1|           6.0|
|   1|2023-01-01|    10|  2|          10.0|
|   1|2023-05-21|    10|  3|           4.0|
|   2|2023-07-28|     2|  1|           1.1|
|   2|2023-08-24|     2|  2|           1.3|
|   2|2023-01-28|     1|  3|           1.0|
+----+----------+------+---+--------------+
only showing top 6 rows



### 2b. El plan — entregable

Ejecuta `explain()` y responde **en esta celda** (edítala) con la pauta de las 4 señales:

1.  **PushedFilters**: ¿qué filtros llegaron hasta Parquet? ¿Falta alguno? ¿por qué? *(todos los filtros de la cláusula `WHERE` del SQL fueron "pushed down" al Parquet Scan, incluyendo `total_amount > 0`, `trip_distance > 0`, y el rango de fechas para `tpep_pickup_datetime`. No falta ninguno, lo que optimiza la lectura al filtrar los datos directamente desde el origen.)*
2.  **ReadSchema**: ¿cuántas columnas lee de las 19? *(Spark lee 4 columnas (`tpep_pickup_datetime`, `trip_distance`, `PULocationID`, `total_amount`) de las 19 columnas originales, como se observa en `ReadSchema: struct<tpep_pickup_datetime:timestamp_ntz,trip_distance:double,PULocationID:int,total_amount:double>`.)*
3.  **Exchange**: ¿cuántos shuffles tiene el plan y qué operación provoca cada uno? *(El plan muestra un `Exchange rangepartitioning` causado por el `ORDER BY zona, pos` final. Además, las operaciones `Window` (`ROW_NUMBER()` y `AVG()` con `PARTITION BY zona`) inherentemente requieren que los datos se agrupen por `zona`, lo que también implica un shuffle o un re-partitioning de los datos para asegurar que todas las filas de una partición se procesen juntas.)*
4.  ¿Qué hizo **AQE** (busca `AdaptiveSparkPlan`)? *(La presencia de `AdaptiveSparkPlan isFinalPlan=false` indica que AQE (Adaptive Query Execution) está activo. AQE permite a Spark optimizar dinámicamente el plan de ejecución de la consulta durante el tiempo de ejecución, basándose en estadísticas reales. Por ejemplo, podría ajustar el número de particiones de shuffle, convertir Sort-Merge Joins en Broadcast Joins o manejar el 'skew' de datos, lo que potencialmente mejora la eficiencia sin requerir configuraciones manuales previas.)*

In [7]:
spark.sql(SQL).explain()


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [zona#767L ASC NULLS FIRST, pos#765 ASC NULLS FIRST], true, 0
   +- Exchange rangepartitioning(zona#767L ASC NULLS FIRST, pos#765 ASC NULLS FIRST, 8), ENSURE_REQUIREMENTS, [plan_id=2516]
      +- Project [zona#767L, fecha#768, viajes#769L, pos#765, round(_we1#773, 1) AS media_movil_7d#766]
         +- Filter (pos#765 <= 3)
            +- Window [avg(viajes#769L) windowspecdefinition(zona#767L, fecha#768 ASC NULLS FIRST, specifiedwindowframe(RowFrame, -6, currentrow$())) AS _we1#773], [zona#767L], [fecha#768 ASC NULLS FIRST]
               +- Sort [zona#767L ASC NULLS FIRST, fecha#768 ASC NULLS FIRST], false, 0
                  +- Window [row_number() windowspecdefinition(zona#767L, viajes#769L DESC NULLS LAST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS pos#765], [zona#767L], [viajes#769L DESC NULLS LAST]
                     +- Sort [zona#767L ASC NULLS FIRST, viajes#769L DESC NULLS LAST], fa

## 3. Motor 2: DuckDB

Mismo SQL, cero cluster: DuckDB consulta los Parquet directamente desde tu proceso.

In [8]:
import duckdb
con = duckdb.connect()
con.sql("CREATE OR REPLACE VIEW viajes AS SELECT * FROM read_parquet('yellow_tripdata_2023-*.parquet', union_by_name=true)")
cronometrar("duckdb", lambda: con.sql(SQL).fetchall())
con.sql(SQL).limit(6).show()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

duckdb {'fria': 3.35, 'mediana': 3.35}


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────┬────────────┬────────┬───────┬────────────────┐
│ zona  │   fecha    │ viajes │  pos  │ media_movil_7d │
│ int64 │    date    │ int64  │ int64 │     double     │
├───────┼────────────┼────────┼───────┼────────────────┤
│     1 │ 2023-07-09 │     11 │     1 │            6.0 │
│     1 │ 2023-01-01 │     10 │     2 │           10.0 │
│     1 │ 2023-06-26 │     10 │     3 │            5.3 │
│     2 │ 2023-08-24 │      2 │     1 │            1.3 │
│     2 │ 2023-07-28 │      2 │     2 │            1.1 │
│     2 │ 2023-01-28 │      1 │     3 │            1.0 │
└───────┴────────────┴────────┴───────┴────────────────┘



In [9]:
# DuckDB también tiene EXPLAIN: compara con el de Spark.
# ¿Ves el equivalente al pushdown (filtro junto al scan)? ¿Y algún 'Exchange'? ¿Por qué no?
con.sql("EXPLAIN " + SQL).show()


┌───────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

## 4. Motor 3: Polars

Misma lógica en la API *lazy* (`scan_parquet` + `collect`): Polars también construye un plan y lo optimiza antes de ejecutar.

In [10]:
import polars as pl

# Mismo truco que en Spark: cada archivo con su esquema, solo las 4 columnas unificadas.
def scan_viajes():
    return pl.concat([
        pl.scan_parquet(f).select(
            pl.col("PULocationID").cast(pl.Int64),
            pl.col("tpep_pickup_datetime").cast(pl.Datetime("us")),
            pl.col("total_amount").cast(pl.Float64),
            pl.col("trip_distance").cast(pl.Float64))
        for f in ARCHIVOS])

def query_polars():
    diarios = (
        scan_viajes()
        .filter((pl.col("total_amount") > 0) & (pl.col("trip_distance") > 0))
        .filter(pl.col("tpep_pickup_datetime").is_between(
            pl.datetime(2023, 1, 1), pl.datetime(2023, 12, 31, 23, 59, 59)))
        .group_by(zona=pl.col("PULocationID"), fecha=pl.col("tpep_pickup_datetime").dt.date())
        .agg(viajes=pl.len())
        .sort(["zona", "fecha"]))
    return (diarios
        .with_columns(
            pos=pl.col("viajes").rank("ordinal", descending=True).over("zona"),
            media_movil_7d=pl.col("viajes").rolling_mean(window_size=7, min_samples=1).over("zona").round(1))
        .filter(pl.col("pos") <= 3)
        .sort(["zona", "pos"])
        .collect())

cronometrar("polars", query_polars)
print(query_polars().head(6))


polars {'fria': 10.36, 'mediana': 4.81}
shape: (6, 5)
┌──────┬────────────┬────────┬─────┬────────────────┐
│ zona ┆ fecha      ┆ viajes ┆ pos ┆ media_movil_7d │
│ ---  ┆ ---        ┆ ---    ┆ --- ┆ ---            │
│ i64  ┆ date       ┆ u32    ┆ u32 ┆ f64            │
╞══════╪════════════╪════════╪═════╪════════════════╡
│ 1    ┆ 2023-07-09 ┆ 11     ┆ 1   ┆ 6.0            │
│ 1    ┆ 2023-01-01 ┆ 10     ┆ 2   ┆ 10.0           │
│ 1    ┆ 2023-05-21 ┆ 10     ┆ 3   ┆ 4.0            │
│ 2    ┆ 2023-07-28 ┆ 2      ┆ 1   ┆ 1.1            │
│ 2    ┆ 2023-08-24 ┆ 2      ┆ 2   ┆ 1.3            │
│ 2    ┆ 2023-01-28 ┆ 1      ┆ 3   ┆ 1.0            │
└──────┴────────────┴────────┴─────┴────────────────┘


In [11]:
# El plan de Polars (el 'explain' de los DataFrames lazy).
# Busca el equivalente a PushedFilters (SELECTION junto al scan) y a la poda de columnas (PROJECT).
plan = (scan_viajes()
        .filter((pl.col("total_amount") > 0) & (pl.col("trip_distance") > 0))
        .group_by("PULocationID").agg(pl.len()))
print(plan.explain())


AGGREGATE[maintain_order: false]
  [len()] BY [col("PULocationID")]
  FROM
  UNION
    PLAN 0:
      simple π 3/3 ["PULocationID", ... 2 other columns]
        Parquet SCAN [yellow_tripdata_2023-01.parquet]
        PROJECT 3/19 COLUMNS
        SELECTION: [([(col("total_amount")) > (0.0)]) & ([(col("trip_distance")) > (0.0)])]
        ESTIMATED ROWS: 3066766
    PLAN 1:
      simple π 3/3 ["PULocationID", ... 2 other columns]
        SELECT [col("PULocationID").cast(Int64), col("total_amount"), col("trip_distance")]
          Parquet SCAN [yellow_tripdata_2023-02.parquet]
          PROJECT 3/19 COLUMNS
          SELECTION: [([(col("trip_distance")) > (0.0)]) & ([(col("total_amount")) > (0.0)])]
          ESTIMATED ROWS: 2913955
    PLAN 2:
      simple π 3/3 ["PULocationID", ... 2 other columns]
        SELECT [col("PULocationID").cast(Int64), col("total_amount"), col("trip_distance")]
          Parquet SCAN [yellow_tripdata_2023-03.parquet]
          PROJECT 3/19 COLUMNS
          SELE

## 5. Tabla de tiempos

**Verifica primero que los tres motores dan el mismo resultado** (mismas filas para una zona conocida, p. ej. la 132 = aeropuerto JFK). Un benchmark con resultados distintos no compara nada.

In [12]:
import pandas as pd
tabla = pd.DataFrame(resultados).T.rename(columns={"fria": "1a corrida (s)", "mediana": "mediana 3 corridas (s)"})
tabla["filas procesadas"] = "~38M"  # ajusta si usaste menos meses
tabla


,1a corrida (s),mediana 3 corridas (s),filas procesadas
spark,37.68,27.46,~38M
duckdb,3.35,3.35,~38M
polars,10.36,4.81,~38M


## 7. Ejercicios de extensión

### E1. Agrega al SQL un JOIN con la tabla de zonas para mostrar el nombre de la zona. En el explain() de Spark: ¿aparece BroadcastHashJoin? ¿Qué lo habilitó?

In [20]:
# Cargar la tabla de zonas en Spark
import urllib.request
import os

zone_lookup_url = 'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv'
zone_lookup_file = 'taxi_zone_lookup.csv'

# Descargar el archivo localmente si no existe
if not os.path.exists(zone_lookup_file):
    urllib.request.urlretrieve(zone_lookup_url, zone_lookup_file)
    print(f"Archivo {zone_lookup_file} descargado.")
else:
    print(f"Archivo {zone_lookup_file} ya existe.")

# Leer el archivo local con Spark
spark_zones_df = spark.read.csv(zone_lookup_file, header=True, inferSchema=True)
spark_zones_df.createOrReplaceTempView('zones')
spark_zones_df.show(3)

Archivo taxi_zone_lookup.csv descargado.
+----------+-------+--------------------+------------+
|LocationID|Borough|                Zone|service_zone|
+----------+-------+--------------------+------------+
|         1|    EWR|      Newark Airport|         EWR|
|         2| Queens|         Jamaica Bay|   Boro Zone|
|         3|  Bronx|Allerton/Pelham G...|   Boro Zone|
+----------+-------+--------------------+------------+
only showing top 3 rows



In [21]:
# Modificar el SQL para incluir el JOIN con la tabla de zonas
SQL_E1 = """
WITH viajes_diarios AS (
  SELECT PULocationID AS zona,
         CAST(tpep_pickup_datetime AS DATE) AS fecha,
         COUNT(*) AS viajes
  FROM viajes
  WHERE total_amount > 0 AND trip_distance > 0
    AND tpep_pickup_datetime >= '2023-01-01' AND tpep_pickup_datetime < '2024-01-01'
  GROUP BY 1, 2
),
ranking_viajes AS (
  SELECT zona, fecha, viajes,
         ROW_NUMBER() OVER (PARTITION BY zona ORDER BY viajes DESC) AS pos,
         ROUND(AVG(viajes) OVER (PARTITION BY zona ORDER BY fecha
               ROWS BETWEEN 6 PRECEDING AND CURRENT ROW), 1) AS media_movil_7d
  FROM viajes_diarios)
SELECT r.zona, z.Zone AS zone_name, r.fecha, r.viajes, r.pos, r.media_movil_7d
FROM ranking_viajes r
JOIN zones z ON r.zona = z.LocationID
WHERE r.pos <= 3
ORDER BY r.zona, r.pos
"""

# Ejecutar la consulta modificada en Spark y mostrar los resultados
spark.sql(SQL_E1).show(6)

+----+--------------+----------+------+---+--------------+
|zona|     zone_name|     fecha|viajes|pos|media_movil_7d|
+----+--------------+----------+------+---+--------------+
|   1|Newark Airport|2023-07-09|    11|  1|           6.0|
|   1|Newark Airport|2023-01-01|    10|  2|          10.0|
|   1|Newark Airport|2023-05-21|    10|  3|           4.0|
|   2|   Jamaica Bay|2023-07-28|     2|  1|           1.1|
|   2|   Jamaica Bay|2023-08-24|     2|  2|           1.3|
|   2|   Jamaica Bay|2023-01-28|     1|  3|           1.0|
+----+--------------+----------+------+---+--------------+
only showing top 6 rows



In [22]:
# Mostrar el plan de ejecución para la consulta modificada
spark.sql(SQL_E1).explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [zona#1048L ASC NULLS FIRST, pos#1051 ASC NULLS FIRST], true, 0
   +- Exchange rangepartitioning(zona#1048L ASC NULLS FIRST, pos#1051 ASC NULLS FIRST, 8), ENSURE_REQUIREMENTS, [plan_id=5684]
      +- Project [zona#1048L, Zone#960 AS zone_name#1047, fecha#1049, viajes#1050L, pos#1051, media_movil_7d#1052]
         +- BroadcastHashJoin [zona#1048L], [cast(LocationID#958 as bigint)], Inner, BuildRight, false
            :- Project [zona#1048L, fecha#1049, viajes#1050L, pos#1051, round(_we1#1056, 1) AS media_movil_7d#1052]
            :  +- Filter (pos#1051 <= 3)
            :     +- Window [avg(viajes#1050L) windowspecdefinition(zona#1048L, fecha#1049 ASC NULLS FIRST, specifiedwindowframe(RowFrame, -6, currentrow$())) AS _we1#1056], [zona#1048L], [fecha#1049 ASC NULLS FIRST]
            :        +- Sort [zona#1048L ASC NULLS FIRST, fecha#1049 ASC NULLS FIRST], false, 0
            :           +- Window [row_number() windowspe

#### Respuesta E1:

1.  **¿Aparece `BroadcastHashJoin`?** Sí, en el plan de ejecución de Spark aparece `BroadcastHashJoin`. Lo podemos ver en las líneas que contienen `... Join BroadcastHashJoin ...`.
2.  **¿Qué lo habilitó?** `BroadcastHashJoin` se habilitó porque la tabla `zones` (la tabla de lookup) es muy pequeña (265 filas). Spark detecta que una de las tablas en el `JOIN` es lo suficientemente pequeña como para ser transmitida (broadcasted) a todos los ejecutores del cluster, lo que permite que cada ejecutor realice el `JOIN` localmente sin necesidad de realizar shuffles de la tabla más grande. Esta es una optimización de rendimiento clave para `JOIN` con tablas dimensionales pequeñas.


### E2. Repite el benchmark con 1 solo mes de datos. ¿Cambia el orden de los motores? ¿Qué te dice eso sobre el costo fijo de cada uno?

In [25]:
# Preparar datos para 1 solo mes (Enero 2023)
ARCHIVOS_E2 = [ARCHIVOS[0]] # Solo el primer archivo, enero 2023
print(ARCHIVOS_E2)

# Reiniciar resultados para E2
resultados_e2 = {}
def cronometrar_e2(motor, fn, corridas=3):
    tiempos = []
    for _ in range(corridas):
        t0 = time.perf_counter(); fn(); tiempos.append(time.perf_counter() - t0)
    resultados_e2[motor] = {"fria": round(tiempos[0], 2), "mediana": round(statistics.median(tiempos), 2)}
    print(motor, resultados_e2[motor])

# --- Spark para 1 mes ---
print("\n--- Ejecutando Spark (1 mes) ---")
partes_e2_spark = [spark.read.parquet(a).select([F.col(c).cast(t).alias(c) for c, t in COLS.items()])
                   for a in ARCHIVOS_E2]
reduce(lambda a, b: a.unionByName(b), partes_e2_spark).createOrReplaceTempView("viajes_e2_spark")
spark.sql("SELECT COUNT(*) AS filas FROM viajes_e2_spark").show()

# SQL para la CTE de Spark en E2
SQL_E2_SPARK_DIARIOS_CTE = """
  CREATE OR REPLACE TEMPORARY VIEW viajes_e2_spark_diarios AS
  SELECT PULocationID AS zona,
         CAST(tpep_pickup_datetime AS DATE) AS fecha,
         COUNT(*) AS viajes
  FROM viajes_e2_spark
  WHERE total_amount > 0 AND trip_distance > 0
    AND tpep_pickup_datetime >= '2023-01-01' AND tpep_pickup_datetime < '2023-02-01' -- Ajustado para enero
  GROUP BY 1, 2
"""

# SQL para la consulta principal de Spark en E2
SQL_E2_SPARK_MAIN = """
SELECT * FROM (
  SELECT zona, fecha, viajes,
         ROW_NUMBER() OVER (PARTITION BY zona ORDER BY viajes DESC) AS pos,
         ROUND(AVG(viajes) OVER (PARTITION BY zona ORDER BY fecha
               ROWS BETWEEN 6 PRECEDING AND CURRENT ROW), 1) AS media_movil_7d
  FROM viajes_e2_spark_diarios) -- Usamos la vista temporal creada
WHERE pos <= 3
ORDER BY zona, pos
"""

def run_spark_e2():
    spark.sql(SQL_E2_SPARK_DIARIOS_CTE) # Creamos la vista temporal de diarios
    spark.sql(SQL_E2_SPARK_MAIN).collect() # Ejecutamos la consulta principal
cronometrar_e2("spark_e2", run_spark_e2)

# --- DuckDB para 1 mes ---
print("\n--- Ejecutando DuckDB (1 mes) ---")
con_e2 = duckdb.connect()
con_e2.sql(f"CREATE OR REPLACE VIEW viajes_e2_duckdb AS SELECT * FROM read_parquet('{ARCHIVOS_E2[0]}', union_by_name=true)")

# SQL para DuckDB en E2 con filtro de fecha ajustado
SQL_E2_DUCKDB = """
WITH viajes_diarios AS (
  SELECT PULocationID AS zona,
         CAST(tpep_pickup_datetime AS DATE) AS fecha,
         COUNT(*) AS viajes
  FROM viajes_e2_duckdb
  WHERE total_amount > 0 AND trip_distance > 0
    AND tpep_pickup_datetime >= '2023-01-01' AND tpep_pickup_datetime < '2023-02-01' -- Ajustado para enero
  GROUP BY 1, 2
)
SELECT * FROM (
  SELECT zona, fecha, viajes,
         ROW_NUMBER() OVER (PARTITION BY zona ORDER BY viajes DESC) AS pos,
         ROUND(AVG(viajes) OVER (PARTITION BY zona ORDER BY fecha
               ROWS BETWEEN 6 PRECEDING AND CURRENT ROW), 1) AS media_movil_7d
  FROM viajes_diarios)
WHERE pos <= 3
ORDER BY zona, pos
"""
def run_duckdb_e2():
    con_e2.sql(SQL_E2_DUCKDB).fetchall()
cronometrar_e2("duckdb_e2", run_duckdb_e2)

# --- Polars para 1 mes ---
print("\n--- Ejecutando Polars (1 mes) ---")
def scan_viajes_e2():
    return pl.concat([
        pl.scan_parquet(f).select(
            pl.col("PULocationID").cast(pl.Int64),
            pl.col("tpep_pickup_datetime").cast(pl.Datetime("us")),
            pl.col("total_amount").cast(pl.Float64),
            pl.col("trip_distance").cast(pl.Float64))
        for f in ARCHIVOS_E2])

def query_polars_e2():
    diarios = (
        scan_viajes_e2()
        .filter((pl.col("total_amount") > 0) & (pl.col("trip_distance") > 0))
        .filter(pl.col("tpep_pickup_datetime").is_between(
            pl.datetime(2023, 1, 1), pl.datetime(2023, 1, 31, 23, 59, 59))) # Este filtro ya es correcto para enero
        .group_by(zona=pl.col("PULocationID"), fecha=pl.col("tpep_pickup_datetime").dt.date())
        .agg(viajes=pl.len())
        .sort(["zona", "fecha"]))
    return (diarios
        .with_columns(
            pos=pl.col("viajes").rank("ordinal", descending=True).over("zona"),
            media_movil_7d=pl.col("viajes").rolling_mean(window_size=7, min_samples=1).over("zona").round(1))
        .filter(pl.col("pos") <= 3)
        .sort(["zona", "pos"])
        .collect())
cronometrar_e2("polars_e2", query_polars_e2)

# Mostrar resultados de E2
tabla_e2 = pd.DataFrame(resultados_e2).T.rename(columns={"fria": "1a corrida (s)", "mediana": "mediana 3 corridas (s)"})
tabla_e2["filas procesadas"] = "~3.4M (1 mes)"
display(tabla_e2)


['yellow_tripdata_2023-01.parquet']

--- Ejecutando Spark (1 mes) ---
+-------+
|  filas|
+-------+
|3066766|
+-------+

spark_e2 {'fria': 4.28, 'mediana': 2.25}

--- Ejecutando DuckDB (1 mes) ---
duckdb_e2 {'fria': 0.51, 'mediana': 0.6}

--- Ejecutando Polars (1 mes) ---
polars_e2 {'fria': 0.42, 'mediana': 0.41}


,1a corrida (s),mediana 3 corridas (s),filas procesadas
spark_e2,4.28,2.25,~3.4M (1 mes)
duckdb_e2,0.51,0.60,~3.4M (1 mes)
polars_e2,0.42,0.41,~3.4M (1 mes)


#### Respuesta E2:

1.  **¿Cambia el orden de los motores?** Sí, el orden y la magnitud de las diferencias entre los motores cambian significativamente cuando se reduce el volumen de datos a un solo mes. Basado en la ejecución con un mes de datos:
    *   **Original (12 meses, ~38M filas):** DuckDB (~3.35s) > Polars (~4.81s) > Spark (~27.46s)
    *   **1 mes (~3.4M filas, Enero 2023):**
        *   **Spark:** **~4.28s** (1a corrida) / **~2.25s** (mediana 3 corridas)
        *   **DuckDB:** **~0.51s** (1a corrida) / **~0.60s** (mediana 3 corridas)
        *   **Polars:** **~0.42s** (1a corrida) / **~0.41s** (mediana 3 corridas)

    El orden relativo de rendimiento de los motores cambia para datos de un solo mes. Mientras que con los 12 meses, DuckDB era el más rápido, seguido de Polars y luego Spark, con un solo mes la eficiencia de **Polars y DuckDB** es aún más notoria, siendo **Polars el más rápido**, seguido de cerca por DuckDB. Spark, aunque mejora su tiempo absoluto, sigue siendo considerablemente más lento. La brecha de rendimiento entre Spark y los otros dos motores se acentúa aún más en términos relativos para volúmenes pequeños.

2.  **¿Qué te dice eso sobre el costo fijo de cada uno?**
    Esta observación resalta el **costo fijo (overhead) de inicialización y orquestación** de cada motor. Spark, al ser un sistema distribuido, tiene un costo fijo considerable asociado con:
    *   El arranque de la JVM y la sesión de Spark.
    *   La inicialización de su planificador y optimizador de consultas, diseñado para entornos a gran escala.
    *   La configuración y coordinación de sus componentes (driver y ejecutores), incluso en modo local.

    Para un volumen de datos pequeño (aproximadamente 3.4 millones de filas), este costo fijo de Spark domina el tiempo total de ejecución. Aunque solo hay un mes de datos, Spark incurre en gran parte de su overhead de inicialización. Por otro lado, DuckDB y Polars, al ser motores in-process y diseñados para eficiencia en una sola máquina, tienen un costo fijo mucho menor. Sus tiempos de ejecución escalan de manera mucho más lineal con el volumen de datos, por lo que para un solo mes, su rendimiento es drásticamente superior al de Spark, y la diferencia en los tiempos es aún más marcada a favor de DuckDB/Polars en comparación con el benchmark de 12 meses. Esto refuerza la idea de que Spark no es la opción más eficiente para volúmenes de datos que pueden ser manejados por una sola máquina.

In [24]:
# Preparar datos para 1 solo mes (Enero 2023)
ARCHIVOS_E2 = [ARCHIVOS[0]] # Solo el primer archivo, enero 2023
print(ARCHIVOS_E2)

# Reiniciar resultados para E2
resultados_e2 = {}
def cronometrar_e2(motor, fn, corridas=3):
    tiempos = []
    for _ in range(corridas):
        t0 = time.perf_counter(); fn(); tiempos.append(time.perf_counter() - t0)
    resultados_e2[motor] = {"fria": round(tiempos[0], 2), "mediana": round(statistics.median(tiempos), 2)}
    print(motor, resultados_e2[motor])

# --- Spark para 1 mes ---
print("\n--- Ejecutando Spark (1 mes) ---")
partes_e2_spark = [spark.read.parquet(a).select([F.col(c).cast(t).alias(c) for c, t in COLS.items()])
                   for a in ARCHIVOS_E2]
reduce(lambda a, b: a.unionByName(b), partes_e2_spark).createOrReplaceTempView("viajes_e2_spark")
spark.sql("SELECT COUNT(*) AS filas FROM viajes_e2_spark").show()

def run_spark_e2():
    spark.sql(SQL.replace('FROM viajes', 'FROM viajes_e2_spark')).collect()
cronometrar_e2("spark_e2", run_spark_e2)

# --- DuckDB para 1 mes ---
print("\n--- Ejecutando DuckDB (1 mes) ---")
con_e2 = duckdb.connect()
con_e2.sql(f"CREATE OR REPLACE VIEW viajes_e2_duckdb AS SELECT * FROM read_parquet('{ARCHIVOS_E2[0]}', union_by_name=true)")

def run_duckdb_e2():
    con_e2.sql(SQL.replace('FROM viajes', 'FROM viajes_e2_duckdb')).fetchall()
cronometrar_e2("duckdb_e2", run_duckdb_e2)

# --- Polars para 1 mes ---
print("\n--- Ejecutando Polars (1 mes) ---")
def scan_viajes_e2():
    return pl.concat([
        pl.scan_parquet(f).select(
            pl.col("PULocationID").cast(pl.Int64),
            pl.col("tpep_pickup_datetime").cast(pl.Datetime("us")),
            pl.col("total_amount").cast(pl.Float64),
            pl.col("trip_distance").cast(pl.Float64))
        for f in ARCHIVOS_E2])

def query_polars_e2():
    diarios = (
        scan_viajes_e2()
        .filter((pl.col("total_amount") > 0) & (pl.col("trip_distance") > 0))
        .filter(pl.col("tpep_pickup_datetime").is_between(
            pl.datetime(2023, 1, 1), pl.datetime(2023, 1, 31, 23, 59, 59)))
        .group_by(zona=pl.col("PULocationID"), fecha=pl.col("tpep_pickup_datetime").dt.date())
        .agg(viajes=pl.len())
        .sort(["zona", "fecha"]))
    return (diarios
        .with_columns(
            pos=pl.col("viajes").rank("ordinal", descending=True).over("zona"),
            media_movil_7d=pl.col("viajes").rolling_mean(window_size=7, min_samples=1).over("zona").round(1))
        .filter(pl.col("pos") <= 3)
        .sort(["zona", "pos"])
        .collect())
cronometrar_e2("polars_e2", query_polars_e2)

# Mostrar resultados de E2
tabla_e2 = pd.DataFrame(resultados_e2).T.rename(columns={"fria": "1a corrida (s)", "mediana": "mediana 3 corridas (s)"})
tabla_e2["filas procesadas"] = "~3.4M (1 mes)"
display(tabla_e2)

['yellow_tripdata_2023-01.parquet']

--- Ejecutando Spark (1 mes) ---
+-------+
|  filas|
+-------+
|3066766|
+-------+



AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `viajes_e2_spark_diarios` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.; line 16 pos 7;
'WithCTE
:- CTERelationDef 18, false
:  +- SubqueryAlias viajes_diarios
:     +- Aggregate [PULocationID#1179L, cast(tpep_pickup_datetime#1180 as date)], [PULocationID#1179L AS zona#1200L, cast(tpep_pickup_datetime#1180 as date) AS fecha#1201, count(1) AS viajes#1202L]
:        +- Filter (((total_amount#1181 > cast(0 as double)) AND (trip_distance#1182 > cast(0 as double))) AND ((tpep_pickup_datetime#1180 >= cast(2023-01-01 as timestamp)) AND (tpep_pickup_datetime#1180 < cast(2024-01-01 as timestamp))))
:           +- SubqueryAlias viajes_e2_spark
:              +- View (`viajes_e2_spark`,

In [16]:
import pandas as pd

print('Validando que los resultados son idénticos para una zona específica (ej. zona 132)...')
spark_result = spark.sql(SQL).filter('zona = 132').toPandas().sort_values(['zona', 'pos']).reset_index(drop=True)
duckdb_result = con.sql(SQL).filter('zona = 132').df().sort_values(['zona', 'pos']).reset_index(drop=True)
polars_result = query_polars().filter(pl.col('zona') == 132).sort(['zona', 'pos']).to_pandas().reset_index(drop=True)

# Estandarizar tipos de columnas para una comparación consistente
# Convertir 'fecha' a datetime64[ns] y normalizar (quitar hora)
spark_result['fecha'] = pd.to_datetime(spark_result['fecha']).astype('datetime64[ns]').dt.normalize()
duckdb_result['fecha'] = pd.to_datetime(duckdb_result['fecha']).astype('datetime64[ns]').dt.normalize()
polars_result['fecha'] = pd.to_datetime(polars_result['fecha']).astype('datetime64[ns]').dt.normalize()

# Estandarizar tipos numéricos (ej. 'pos' a int64, 'viajes' a int64, 'media_movil_7d' a float64)
spark_result['zona'] = spark_result['zona'].astype('int64')
duckdb_result['zona'] = duckdb_result['zona'].astype('int64')
polars_result['zona'] = polars_result['zona'].astype('int64')

spark_result['viajes'] = spark_result['viajes'].astype('int64')
duckdb_result['viajes'] = duckdb_result['viajes'].astype('int64')
polars_result['viajes'] = polars_result['viajes'].astype('int64')

spark_result['pos'] = spark_result['pos'].astype('int64')
duckdb_result['pos'] = duckdb_result['pos'].astype('int64')
polars_result['pos'] = polars_result['pos'].astype('int64')

spark_result['media_movil_7d'] = spark_result['media_movil_7d'].astype('float64')
duckdb_result['media_movil_7d'] = duckdb_result['media_movil_7d'].astype('float64')
polars_result['media_movil_7d'] = polars_result['media_movil_7d'].astype('float64')

# Comparar DataFrames
if spark_result.equals(duckdb_result) and spark_result.equals(polars_result):
    print("¡Validación exitosa! Los resultados para la zona 132 son idénticos en Spark, DuckDB y Polars.")
    display(spark_result) # Mostrar uno de los resultados para confirmación
else:
    print("¡Advertencia! Los resultados para la zona 132 NO son idénticos. Detalles a continuación:")
    print("\n--- Dtypes Spark ---"); display(spark_result.dtypes)
    print("\n--- Dtypes DuckDB ---"); display(duckdb_result.dtypes)
    print("\n--- Dtypes Polars ---"); display(polars_result.dtypes)
    print("\n--- Spark result ---"); display(spark_result)
    print("\n--- DuckDB result ---"); display(duckdb_result)
    print("\n--- Polars result ---"); display(polars_result)

    # Try to find the exact differences if they are not equal
    if not spark_result.equals(duckdb_result):
        print("\n--- Diferencias Spark vs DuckDB ---")
        display(spark_result.compare(duckdb_result))
    if not spark_result.equals(polars_result):
        print("\n--- Diferencias Spark vs Polars ---")
        display(spark_result.compare(polars_result))


Validando que los resultados son idénticos para una zona específica (ej. zona 132)...
¡Validación exitosa! Los resultados para la zona 132 son idénticos en Spark, DuckDB y Polars.


,zona,fecha,viajes,pos,media_movil_7d
0,132,2023-09-05,7288,1,6058.0
1,132,2023-11-27,7267,2,5043.6
2,132,2023-09-04,7160,3,5818.9


## Conclusión Técnica: DuckDB vs. Spark en este Benchmark

En este benchmark específico, DuckDB superó a Spark de manera significativa debido a varias razones técnicas clave que se derivan de sus arquitecturas fundamentales y el contexto del problema:

1.  **Naturaleza In-Process y Single-Node vs. Distribuida:** DuckDB es una base de datos OLAP (Online Analytical Processing) embebida y optimizada para operar en una única máquina, directamente dentro del proceso de la aplicación Python. Esto elimina por completo la sobrecarga asociada con la coordinación de procesos distribuidos, la serialización/deserialización de datos para la transferencia de red, y la gestión de fallos que es inherente a Spark.
2.  **Overhead de Spark:** Spark, incluso en modo local (local[*]), introduce un overhead considerable debido a su diseño distribuido. Debe inicializar un 'mini-cluster' (driver y executors en la misma JVM), gestionar el ciclo de vida de las tareas, y aplicar su planificador y optimizador de consultas con la complejidad de un entorno distribuido en mente. Para un dataset de ~700 MB, esta sobrecarga domina el tiempo de ejecución.
3.  **Eficiencia de Acceso a Datos y Computación:** DuckDB está diseñado para un acceso extremadamente eficiente a datos locales (como archivos Parquet en disco) y para realizar computaciones vectorizadas y columnares directamente en memoria, minimizando copias y movimientos de datos. Su optimizador de consultas está afinado para este escenario, aprovechando al máximo los recursos de una sola CPU y RAM.
4.  **Tamaño del Dataset:** El volumen de datos (~38 millones de filas, ~700 MB) es lo suficientemente pequeño como para ser procesado eficientemente en la memoria de una máquina moderna. Esto significa que la principal ventaja de Spark (su capacidad para escalar a terabytes o petabytes de datos en un cluster) no entra en juego aquí. Para estos volúmenes, la eficiencia de un motor local y optimizado para OLAP como DuckDB es superior.

En resumen, mientras que Spark es la herramienta indiscutible para big data distribuido, DuckDB brilla en escenarios donde los datos son "grandes" pero manejables por una sola máquina, ofreciendo una performance superior al evitar la complejidad y el overhead de un sistema distribuido.

## 6. Recomendación en 3 líneas — entregable

Con tu tabla de tiempos, responde **como si se lo explicaras a un gerente que paga la boleta cloud**:

1.  Para *este* volumen (~38M filas, ~700 MB) y *esta* consulta, ¿qué motor recomiendas y por qué? *(Según los resultados del benchmark, para este volumen de datos y esta consulta específica, recomiendo encarecidamente **DuckDB**. Su tiempo de ejecución mediano fue de 3.35 segundos, superando significativamente a Polars (4.81 segundos) y Spark (27.46 segundos). DuckDB ofrece la mejor performance en un entorno de máquina única para esta tarea.)*
2.  ¿En qué punto cambiaría tu recomendación? (volumen, crecimiento, si el pipeline ya vive en un lakehouse, streaming...) *(Mi recomendación cambiaría si el volumen de datos creciera a un punto en el que ya no pueda ser manejado eficientemente por una sola máquina (ej. cientos de GBs o TBs), o si la infraestructura existente del pipeline ya estuviera basada en un lakehouse o sistema de streaming que utilice Spark. En esos escenarios, Spark se convertiría en la opción preferible debido a su escalabilidad distribuida y su ecosistema integral, a pesar de ser más lento para este volumen particular. También, si la tolerancia a fallos distribuida se volviera una prioridad, Spark sería la elección.)*
3.  ¿Qué costo tiene tu opción ganadora que la tabla de tiempos **no** muestra? *(La tabla de tiempos no muestra el **uso de recursos de memoria (RAM)** de DuckDB. Aunque es muy rápido, DuckDB procesa los datos en memoria en una única máquina, lo que podría convertirse en un cuello de botella para datasets mucho más grandes que no quepan en la RAM disponible. También hay un "costo" de curva de aprendizaje si el equipo no está familiarizado con DuckDB y la gestión de dependencias en un entorno de producción si se integra con otros sistemas no nativos de un lakehouse.)*

## 7. Ejercicios de extensión (si el tiempo alcanza)

**E1.** Agrega al SQL un `JOIN` con la tabla de zonas (`https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv`, 265 filas) para mostrar el nombre de la zona. En el `explain()` de Spark: ¿aparece `BroadcastHashJoin`? ¿Qué lo habilitó?

**E2.** Repite el benchmark con **1 solo mes** de datos. ¿Cambia el orden de los motores? ¿Qué te dice eso sobre el costo fijo de cada uno?

**E3.** Quita el `PARTITION BY zona` de la ventana en Spark y compara los planes: ¿qué advertencia esperarías y por qué es peligrosa a escala?

## 8. Declaración de uso de IA — obligatoria

Indica aquí si usaste asistentes de IA (text-to-SQL, autocompletado, chat): **en qué celdas, con qué prompt y cómo validaste el resultado** (las 3 preguntas de la clase: ¿responde la pregunta real?, ¿validaste contra números conocidos?, ¿revisaste el plan?). Si no usaste, escribe "Sin uso de IA".

---
**Entrega:** notebook ejecutado de arriba a abajo (Kernel → Restart & Run All antes de subir), con las celdas-respuesta de las secciones 2b y 6 completas.

Indica aquí si usaste asistentes de IA (text-to-SQL, autocompletado, chat): **en qué celdas, con qué prompt y cómo validaste el resultado** (las 3 preguntas de la clase: ¿responde la pregunta real?, ¿validaste contra números conocidos?, ¿revisaste el plan?). Si no usaste, escribe "Sin uso de IA".

---

Se utilizó asistencia de IA para la generación y refinamiento de las respuestas en las celdas:

*   **Celda `ThoPtIhta4Yj` (Interpretación del plan de Spark):** Se solicitó asistencia para interpretar las '4 señales' (PushedFilters, ReadSchema, Exchange, AQE) del plan de ejecución de Spark. Validé el resultado comparándolo con la salida de `spark.sql(SQL).explain()` y asegurándome de que cada punto se correspondiera con la información del plan.
*   **Celda `2jCCLDDqa4Yr` (Recomendación):** Se solicitó ayuda para estructurar la recomendación, considerando el público objetivo (un gerente que paga la boleta cloud) y los puntos clave solicitados (qué motor, cuándo cambiaría la recomendación, costos no visibles). Validé que la recomendación se basara estrictamente en los tiempos obtenidos en la `tabla` (celda `D50YOMrla4Yr`) y que las justificaciones fueran coherentes con las características de cada motor.
*   **Celda `a5289b4f` (Conclusión Técnica):** Se utilizó para elaborar una conclusión técnica sobre por qué DuckDB supera a Spark en este benchmark específico. Validé la exactitud de las explicaciones sobre las arquitecturas de ambos motores y cómo sus características impactan el rendimiento para el volumen de datos dado.

En todos los casos, la validación se realizó cotejando la información generada con los datos reales del benchmark (tiempos y planes de ejecución), el contenido del notebook, y mi conocimiento técnico sobre Spark, DuckDB y Polars para asegurar la precisión y relevancia de las respuestas. Además, se agregó una validación explícita de los resultados de la consulta para una zona específica (celda insertada antes de la sección 5) para confirmar la consistencia entre los motores.